# Task-1 Week-3.

**Dataset :-** 'Admission_Prediction.csv'

**Objective :-** Apply chi-square tests to rigorously assess associations between categorical applicant features (such as program choice and nationality) and admission outcomes, guiding feature selection.

In [34]:
# import library

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [35]:
# Load the dataset

df = pd.read_csv("Admission_Prediction.csv")

print("DATASET SHAPE:", df.shape)
print("\nFIRST FIVE ROWS:")
print(df.head())

print("\nMISSING VALUES:")
print(df.isnull().sum())


DATASET SHAPE: (1000, 13)

FIRST FIVE ROWS:
   Serial No.  GRE Score  TOEFL Score  University Rating  SOP  LOR  CGPA  \
0           1        337          118                  4  4.5  4.5  9.65   
1           2        324          107                  4  4.0  4.5  8.87   
2           3        316          104                  3  3.0  3.5  8.00   
3           4        322          110                  3  3.5  2.5  8.67   
4           5        314          103                  2  2.0  3.0  8.21   

   Research  Chance of Admit  Year  \
0         1             0.92  2022   
1         1             0.76  2022   
2         1             0.72  2022   
3         1             0.80  2022   
4         0             0.65  2022   

                              University_Name region Admission Status  
0    Indian Institute of Technology Kharagpur   East         Accepted  
1  National Institute of Technology Karnataka  South         Accepted  
2                          Gujarat University   West  

In [36]:
# Chi Square feature analysis

df["Chance_Category"] = np.where(
    df["Chance of Admit"] >= 0.75,
    "Higher Chance",
    "Lower Chance"
)

features = [
    "Research",
    "University Rating",
    "region",
    "Year",
    "University_Name"
]

results = []

for feature in features:

    # Create contingency table
    table = pd.crosstab(
        df[feature],
        df["Chance_Category"]
    )

    # Perform Chi-Square Test of Independence
    chi2, p_value, dof, expected = chi2_contingency(table)

    # Check expected cell frequencies
    cells_below_5 = int((expected < 5).sum())

    # Interpret p-value
    if p_value < 0.05:
        decision = "Significant association"
    else:
        decision = "No significant association"

    results.append({
        "Feature": feature,
        "Chi-Square": round(chi2, 4),
        "P-Value": p_value,
        "Degrees of Freedom": dof,
        "Expected Cells < 5": cells_below_5,
        "Decision": decision
    })
chi_results = pd.DataFrame(results)

print("\nCHI-SQUARE RESULTS")
print(chi_results.to_string(index=False))

# Display detailed contingency table
print("\nRESEARCH VS CHANCE CATEGORY:")
print(pd.crosstab(df["Research"], df["Chance_Category"]))



CHI-SQUARE RESULTS
          Feature  Chi-Square      P-Value  Degrees of Freedom  Expected Cells < 5                   Decision
         Research     57.9981 2.623699e-14                   1                   0    Significant association
University Rating     57.2313 1.106384e-11                   4                   0    Significant association
           region      8.6779 1.226216e-01                   5                   0 No significant association
             Year      1.6275 8.038404e-01                   4                   0 No significant association
  University_Name     48.1772 4.656738e-01                  48                   5 No significant association

RESEARCH VS CHANCE CATEGORY:
Chance_Category  Higher Chance  Lower Chance
Research                                    
0                          156           341
1                          279           224


In [37]:
# define target variable and input features

target = "Chance of Admit"

# Remove target and serial number from input features
X = df.drop(
    columns=[target, "Serial No.", "Chance_Category"],
    errors="ignore"
)

y = df[target]

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

print("\nNUMERICAL FEATURES:", numerical_cols)
print("CATEGORICAL FEATURES:", categorical_cols)


NUMERICAL FEATURES: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Year']
CATEGORICAL FEATURES: ['University_Name', 'region', 'Admission Status']


In [38]:
# prepare preprocessing pipelines for numerical and categorical features

# Numerical features: replace missing values with median
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical features: fill missing values and encode
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

In [39]:
# create a Random forest regression model

model = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ))
])


In [40]:
# split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTRAINING RECORDS:", len(X_train))
print("TESTING RECORDS:", len(X_test))


TRAINING RECORDS: 800
TESTING RECORDS: 200


In [41]:
# train the model

model.fit(X_train, y_train)

print("\nModel training completed successfully.")


y_pred = model.predict(X_test)



Model training completed successfully.


In [42]:
# evualuate the model performance

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("\nMODEL PERFORMANCE")
print("Mean Absolute Error (MAE):", round(mae, 4))
print("Root Mean Squared Error (RMSE):", round(rmse, 4))
print("R-squared (R2):", round(r2, 4))



MODEL PERFORMANCE
Mean Absolute Error (MAE): 0.0622
Root Mean Squared Error (RMSE): 0.0788
R-squared (R2): 0.6382


In [43]:
# five-fold cross-validation 

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="r2"
)

print("\nCROSS-VALIDATION")
print("R2 scores:", np.round(cv_scores, 4))
print("Average CV R2:", round(cv_scores.mean(), 4))


CROSS-VALIDATION
R2 scores: [0.6353 0.6761 0.6406 0.6383 0.6633]
Average CV R2: 0.6507


In [45]:
# predicting the chance of admit for a new student

new_student = pd.DataFrame([{
    "GRE Score": 325,
    "TOEFL Score": 110,
    "University Rating": 4,
    "SOP": 4.0,
    "LOR": 4.0,
    "CGPA": 9.0,
    "Research": 1,
    "Year": 2026,
    "University_Name": "Gujarat University",
    "region": "West",
    "Admission Status": "Admitted"
}])

predicted_chance = model.predict(new_student)[0]

print("\n NEW STUDENT PREDICTION ")
print("Predicted Chance of Admit:", round(predicted_chance, 4))
print("Predicted Chance (%):", round(predicted_chance * 100, 2), "%")



 NEW STUDENT PREDICTION 
Predicted Chance of Admit: 0.8134
Predicted Chance (%): 81.34 %
